# ESM3 FerroCLF — v4 (external validation: positive + hard-neg only)

Same model and training as the hard-negative random-split run (housekeeping stays
in the **training** set — that is what yields the 0.6294 hard-negative specificity),
but external validation reports **positive + hard-negative (death genes) only** — the
easy-negative / housekeeping tier is dropped. Retrains from scratch, then evaluates.

**Run:** set Runtime → GPU, add a valid `HF_TOKEN` Colab secret, Run all.
Outputs are written to `MyDrive/JR_Ferro/v4/`.


## A · Setup, install, HuggingFace auth

In [ ]:
# esm is needed to embed the new sequences. Installing it here; no restart required
# (we do NOT force a numpy 2.x upgrade, which is what made the package-test notebook
# need a restart).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'esm', 'openpyxl', 'biopython', 'h5py', 'scikit-learn'], check=False)
print('Install step done.')


In [ ]:
import math, time, re, gc, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score

from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein
from esm.tokenization.sequence_tokenizer import EsmSequenceTokenizer

from google.colab import drive
drive.mount('/content/drive')

# HuggingFace auth (ESM3 is gated)
from huggingface_hub import login, whoami
try:
    whoami(); print('HuggingFace: already authenticated')
except Exception:
    try:
        from google.colab import userdata
        login(userdata.get('HF_TOKEN'))       # HF_TOKEN Colab secret (Runtime → Manage secrets)
        print('HuggingFace: logged in via Colab secret HF_TOKEN')
    except Exception:
        login()                                # interactive fallback

# Reproducibility / device
random_seed = 42
torch.manual_seed(random_seed); np.random.seed(random_seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device  = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
use_amp = (device.type == 'cuda')
print(f'Device: {device}')


In [ ]:
# Paths & config
PROJECT_ROOT = Path('/content/drive/MyDrive/JR_Ferro')
EMBED_DIR    = PROJECT_ROOT / 'ESM3_Embedding'
ESM2_DIR     = PROJECT_ROOT / 'ESM_Embedding'
EXTERNAL_DIR = PROJECT_ROOT / 'Data/Ferro/external'     # ferroptosis genes (held-out positives)
PR_PATH      = EMBED_DIR / 'esm3_per_residue.h5'        # original cached embeddings

_local = Path('/mnt/local-scratch/esm3_per_residue.h5')
if _local.exists():
    PR_PATH = _local; print(f'Using NVMe copy: {PR_PATH}')

HARD_PATH   = EMBED_DIR / 'esm3_hardneg_train.h5'       # cache: train hard-neg embeddings
EXTVAL_PATH = EMBED_DIR / 'esm3_extval_holdout.h5'      # cache: held-out validation embeddings
CKPT_PATH   = EMBED_DIR / 'model_checkpoints/best_esm3ferrocif_v4.pth'
V4_DIR      = PROJECT_ROOT / 'v4'; V4_DIR.mkdir(parents=True, exist_ok=True)  # all v4 outputs

NEG_DIR = EXTERNAL_DIR / 'negative'   # the folder the ORIGINAL external validation used
                                      # (= RCD's genes + FADD). Single negative source.

# Knobs (edit if you want)
PER_GENE_CAP     = 1500
HOLDOUT_FRAC     = 0.30
# Drop from NEGATIVES: NLRP3 (ambiguous), FTL + YY1AP1 (actually ferroptosis
# positives — this is the mislabeling fix; they stay in external/ as positives).
EXCLUDE_NEG      = {'NLRP3', 'FTL', 'YY1AP1'}
EXCLUDE_POS      = {'NLRP3'}           # NLRP3 fully dropped (also not used as positive)
MAX_SEQ_LEN      = 1536
EMBED_BATCH      = 8
VALID_AA         = set('ACDEFGHIKLMNPQRSTVWY')

# Pathway map (external/negative/ is a flat folder with no pathway labels, so we
# supply them here — same grouping as the RCD/ subfolders, plus FADD → apoptosis).
PATHWAY = {
    'ANXA5':'Apoptosis','APAF1':'Apoptosis','BBC3':'Apoptosis','BCL2':'Apoptosis',
    'CASP3':'Apoptosis','CASP7':'Apoptosis','DFFB':'Apoptosis','FADD':'Apoptosis',
    'CASP8':'Necroptosis','MLKL':'Necroptosis','RIPK1':'Necroptosis','RIPK3':'Necroptosis',
    'TNFRSF1A':'Necroptosis','TRADD':'Necroptosis','ZBP1':'Necroptosis',
    'AIM2':'Pyroptosis','CASP1':'Pyroptosis','CASP4':'Pyroptosis','CASP5':'Pyroptosis',
    'GSDMD':'Pyroptosis','GSDME':'Pyroptosis','IL1B':'Pyroptosis',
}

for p in [PR_PATH, EXTERNAL_DIR]:
    assert Path(p).exists(), f'Missing: {p}'
assert NEG_DIR.exists(), (
    f'Negative folder not found: {NEG_DIR}\n'
    'The original external validation read negatives from external/negative/.\n'
    'If it is no longer on your Drive, either restore it, or fall back to the\n'
    'RCD/ folder (upload it and point NEG_DIR at Data/Ferro/RCD with glob "*/*.xlsx").')
print('Paths OK.')


## B · Parse hard negatives from `external/negative/`, cap, split 70 / 30

Single negative source: the folder the original external validation used
(= the RCD death genes **plus FADD**). FTL and YY1AP1 are skipped here — they are
ferroptosis positives and stay in `external/`. Each gene is capped at
`PER_GENE_CAP`; genes are split **by gene** (never by sequence), 70/30 within each
pathway so both the train pool and the held-out set span all three death types.

In [ ]:
def gene_from_fname(name):
    s = re.sub(r'\.xlsx$', '', name)      # strip up to two .xlsx (files use .xlsx.xlsx)
    s = re.sub(r'\.xlsx$', '', s)
    s = re.sub(r'^uniparc_', '', s)
    s = re.split(r'_AND_|_20\d\d', s)[0]  # cut query/date suffix
    return s.strip('_')


def read_xlsx_any(path):
    """Read an xlsx whether it is plain OOXML or gzip-wrapped (the RCD/ files are
    gzip-compressed; the external/ files are plain — this handles both)."""
    import gzip, io
    raw = Path(path).read_bytes()
    if raw[:2] == b'\x1f\x8b':
        raw = gzip.decompress(raw)
    return pd.read_excel(io.BytesIO(raw), engine='openpyxl')


def parse_xlsx(path, label, pathway):
    df = read_xlsx_any(path)
    df.columns = [c.strip() for c in df.columns]
    seq_col = next((c for c in df.columns if 'seq'   in c.lower()), df.columns[1])
    ent_col = next((c for c in df.columns if 'entry' in c.lower()), df.columns[0])
    gene = gene_from_fname(path.name)
    rows = []
    for _, r in df.iterrows():
        seq = ''.join(c for c in str(r[seq_col]).strip().upper() if c in VALID_AA)
        if len(seq) >= 10:
            rows.append({'gene': gene, 'pathway': pathway,
                         'entry': str(r[ent_col]).strip(), 'sequence': seq, 'label': label})
    return rows


# Read the flat external/negative/ folder
records, unknown_pw = [], set()
for path in sorted(NEG_DIR.glob('*.xlsx')):
    g = gene_from_fname(path.name)
    if g in EXCLUDE_NEG:
        continue
    pw = PATHWAY.get(g)
    if pw is None:
        unknown_pw.add(g); pw = 'Other'   # surfaced below, still usable
    records.extend(parse_xlsx(path, label=0, pathway=pw))

rcd = pd.DataFrame(records)
print(f'Negative genes found in external/negative/: {sorted(rcd["gene"].unique())}')
print(f'Parsed {len(rcd)} negative sequences across {rcd["gene"].nunique()} genes')
print(rcd.groupby('pathway')['gene'].nunique().to_string())
has_fadd = 'FADD' in set(rcd['gene'])
print(f'FADD present (capped hard negative): {has_fadd}')
if unknown_pw:
    print(f'\n*** Genes with no pathway in PATHWAY map (check these): {sorted(unknown_pw)}')

# Per-gene cap (deterministic sample)
rng = np.random.default_rng(random_seed)
def cap(group):
    if len(group) <= PER_GENE_CAP: return group
    return group.iloc[rng.permutation(len(group))[:PER_GENE_CAP]]
rcd = rcd.groupby('gene', group_keys=False).apply(cap).reset_index(drop=True)
print(f'After cap ({PER_GENE_CAP}/gene): {len(rcd)} sequences')

# Gene-disjoint 70/30, stratified by pathway
gene_pw = rcd[['gene', 'pathway']].drop_duplicates().sort_values('gene').reset_index(drop=True)
holdout_genes, train_genes = [], []
for pw, sub in gene_pw.groupby('pathway'):
    genes = sub['gene'].to_numpy()
    order = np.random.default_rng(random_seed).permutation(len(genes))
    n_hold = max(1, round(len(genes) * HOLDOUT_FRAC))
    holdout_genes += list(genes[order[:n_hold]])
    train_genes   += list(genes[order[n_hold:]])
train_genes, holdout_genes = set(train_genes), set(holdout_genes)
assert not (train_genes & holdout_genes)

rcd_train = rcd[rcd['gene'].isin(train_genes)].reset_index(drop=True)
rcd_hold  = rcd[rcd['gene'].isin(holdout_genes)].reset_index(drop=True)
print(f'\nTrain-pool hard-neg genes ({len(train_genes)}): {sorted(train_genes)}')
print(f'Held-out  hard-neg genes ({len(holdout_genes)}): {sorted(holdout_genes)}')
print(f'Train-pool seqs: {len(rcd_train)}   Held-out seqs: {len(rcd_hold)}')


## C · Embed the train-pool hard negatives with ESM3

Same recipe as the original cache (`embeddings[:, 1:L+1, :]`, float16), so these
are drop-in compatible with `esm3_per_residue.h5`. Cached — reruns skip this.

In [ ]:
print('Loading ESM3 (esm3_sm_open_v1) ...', flush=True)
esm3_model = ESM3.from_pretrained('esm3_sm_open_v1').to(torch.bfloat16).to(device).eval()
_tok    = EsmSequenceTokenizer()
_pad_id = _tok.pad_token_id
print(f'ESM3 ready (dtype={next(esm3_model.parameters()).dtype})')


def embed_to_h5(seqs, keys, out_path, batch=EMBED_BATCH):
    """Per-residue ESM3 embeddings → HDF5 keyed by `keys` (float16, gzip). BOS stripped."""
    out_path = Path(out_path)
    order = np.argsort([len(s) for s in seqs])   # length-sort to minimise padding
    with h5py.File(out_path, 'w') as hf:
        i, t0 = 0, time.time()
        while i < len(order):
            sel = order[i:i + batch]
            bs_seqs = [seqs[j] for j in sel]
            try:
                toks, lens = [], []
                for s in bs_seqs:
                    pt = esm3_model.encode(ESMProtein(sequence=s[:MAX_SEQ_LEN]))
                    toks.append(pt.sequence); lens.append(min(len(s), MAX_SEQ_LEN))
                mx = max(t.shape[0] for t in toks)
                padded = torch.stack([F.pad(t, (0, mx - t.shape[0]), value=_pad_id)
                                      for t in toks]).to(device)
                with torch.inference_mode():
                    with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16):
                        out = esm3_model(sequence_tokens=padded)
                for k, j in enumerate(sel):
                    L = lens[k]
                    emb = out.embeddings[k, 1:L + 1, :].float().cpu().numpy().astype('float16')
                    hf.create_dataset(str(keys[j]), data=emb, compression='gzip', compression_opts=4)
                i += len(sel)
                if i % 200 < batch or i >= len(order):
                    print(f'  {min(i,len(order))}/{len(order)}  ({time.time()-t0:.0f}s)', flush=True)
            except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                if 'out of memory' not in str(e).lower(): raise
                torch.cuda.empty_cache(); gc.collect()
                batch = max(1, batch // 2)
                print(f'  OOM → batch {batch}', flush=True)
    print(f'Saved {out_path}  ({out_path.stat().st_size/1e6:.0f} MB)')


# keys for hard-neg train rows: 'H0','H1',... (namespaced, never collide with orig ids)
rcd_train = rcd_train.copy()
rcd_train['h5key'] = ['H' + str(i) for i in range(len(rcd_train))]

if HARD_PATH.exists():
    print(f'Reusing cached {HARD_PATH}')
else:
    embed_to_h5(rcd_train['sequence'].tolist(), rcd_train['h5key'].tolist(), HARD_PATH)
rcd_train[['gene','pathway','entry','label','h5key']].to_csv(
    EMBED_DIR / 'hardneg_train_meta.csv', index=False)


## D · Combine with original data, RANDOM (sequence-level) split, train

**This is the deliberate difference from the gene-disjoint notebook.** Here the
combined data is split by *sequence*, stratified by label (the original method).
Sequence-variants of the same gene therefore appear in both train and test — so
the internal test number reflects **classifying new variants of KNOWN genes**, not
generalising to novel genes. Read it as a scoped practical tool, and read the
Section E external validation (fully held-out genes) as the honest novel-gene test.

In [ ]:
# Original aligned dataset (identical to the other notebooks)
esm3_meta = pd.read_csv(EMBED_DIR / 'sequence_metadata.csv')
esm2_meta = pd.read_csv(ESM2_DIR  / 'sequence_metadata.csv')
merged = esm3_meta.merge(
    esm2_meta[['header','sequence_id']].rename(columns={'sequence_id':'esm2_idx'}),
    on='header', how='inner')
orig_gene = merged['gene'].astype(str).str.split('_AND_').str[0].to_numpy()
print(f'Original aligned: {len(merged)}  ({merged["label"].mean():.3f} pos)')

# Combined index: one row per training sequence, from either H5 source
COMB = pd.concat([
    pd.DataFrame({'src':'orig', 'key':merged['sequence_id'].to_numpy(),
                  'label':merged['label'].to_numpy(), 'gene':orig_gene}),
    pd.DataFrame({'src':'hard', 'key':rcd_train['h5key'].to_numpy(),
                  'label':rcd_train['label'].to_numpy(), 'gene':rcd_train['gene'].to_numpy()}),
], ignore_index=True)
PATHS = {'orig': str(PR_PATH), 'hard': str(HARD_PATH)}
COMB_SRC, COMB_KEY = COMB['src'].to_numpy(), COMB['key'].to_numpy()
COMB_Y, COMB_GENE  = COMB['label'].to_numpy(dtype=np.int64), COMB['gene'].to_numpy()

# guard: no hard-neg gene accidentally shares a symbol with an original gene
_overlap = set(rcd_train['gene']) & set(orig_gene)
assert not _overlap, f'hard-neg genes already in original data: {_overlap}'
# guard: held-out genes are NOT in the training pool
assert not (set(holdout_genes) & set(COMB_GENE)), 'held-out gene leaked into training pool'
print(f'Combined training rows: {len(COMB)}  ({COMB_Y.mean():.3f} pos, '
      f'{len(rcd_train)} hard negatives added)')

# Hold out housekeeping (EASY-negative) genes for external validation
# These are removed from training entirely, so their external specificity is a
# fair held-out number (not leakage). Their embeddings are already in the orig cache.
HK_HOLDOUT = {'SKP1','FEN1','RPL21','TBP','GAPDH','PPIA','RPS2','EXO1','DES',
              'RDX','FLNA','ALDOA','PGK1','HSPA8','MAPT'}
hk_avail = sorted(HK_HOLDOUT & set(COMB_GENE[COMB_Y == 0]))
hk_pos   = np.where(np.isin(COMB_GENE, hk_avail) & (COMB_Y == 0))[0]
print(f'\nHousekeeping held out (easy negatives): {hk_avail}')
print(f'  → {len(hk_pos)} sequences from {len(hk_avail)} genes removed from training')

# Random stratified split on the REMAINING rows (original method): 64/16/20
from sklearn.model_selection import train_test_split
_pool = np.setdiff1d(np.arange(len(COMB_Y)), hk_pos)
tv, idx_test = train_test_split(_pool, test_size=0.20,
                                stratify=COMB_Y[_pool], random_state=random_seed)
idx_train, idx_val = train_test_split(tv, test_size=0.20,
                                      stratify=COMB_Y[tv], random_state=random_seed)
assert not (set(hk_pos) & set(np.concatenate([idx_train, idx_val, idx_test]))), \
    'housekeeping holdout leaked into the split'
y_train, y_val, y_test = COMB_Y[idx_train], COMB_Y[idx_val], COMB_Y[idx_test]

# NOTE: genes are intentionally SHARED across splits here (that is the point).
print('\nRandom (sequence-level) split')
for nm, idx in [('Train',idx_train), ('Val',idx_val), ('Test',idx_test)]:
    n_g = len(set(COMB_GENE[idx]))
    print(f'  {nm:5s}: {len(idx):6d} seqs  {n_g:3d} genes  {COMB_Y[idx].mean():.3f} pos')
_shared = len(set(COMB_GENE[idx_train]) & set(COMB_GENE[idx_test]))
print(f'  Genes shared train∩test: {_shared}  (expected — variants leak; that is the design)')


In [ ]:
# Model / dataset / loader (multi-H5 aware)
class MultiH5Dataset(torch.utils.data.Dataset):
    def __init__(self, srcs, keys, labels, paths, max_len):
        self.srcs, self.keys = srcs, keys
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.paths, self.max_len, self._h = paths, max_len, {}
    def _f(self, src):
        if src not in self._h: self._h[src] = h5py.File(self.paths[src], 'r')
        return self._h[src]
    def __len__(self): return len(self.keys)
    def __getitem__(self, i):
        emb = self._f(self.srcs[i])[str(self.keys[i])][:].astype('float32')
        if emb.shape[0] > self.max_len: emb = emb[:self.max_len]
        return torch.from_numpy(emb), self.labels[i]


def collate_variable_length(batch):
    seqs, labels = zip(*batch)
    lengths = [s.shape[0] for s in seqs]
    max_L, D = max(lengths), seqs[0].shape[1]
    padded = torch.zeros(len(seqs), max_L, D)
    mask   = torch.ones(len(seqs), max_L, dtype=torch.bool)
    for i, (s, L) in enumerate(zip(seqs, lengths)):
        padded[i, :L] = s; mask[i, :L] = False
    return padded, torch.stack(list(labels)), mask


class SinusoidalPosEnc(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe  = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(pos * div); pe[0, :, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)
    def forward(self, x): return x + self.pe[:, :x.size(1)]


class ESM3FerroCLF(nn.Module):
    def __init__(self, input_dim=1536, proj_dim=256, num_layers=4,
                 num_heads=8, ffn_dim=512, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim)
        self.pos_enc = SinusoidalPosEnc(proj_dim)
        self.norm_in = nn.LayerNorm(proj_dim)
        enc = nn.TransformerEncoderLayer(d_model=proj_dim, nhead=num_heads,
            dim_feedforward=ffn_dim, dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers,
            enable_nested_tensor=False)
        self.attn_q  = nn.Parameter(torch.randn(proj_dim) * 0.02)
        self.norm_out = nn.LayerNorm(proj_dim)
        self.drop = nn.Dropout(dropout); self.fc = nn.Linear(proj_dim, 2)
    def forward(self, x, padding_mask=None, return_weights=False):
        h = self.norm_in(self.proj(x)); h = self.pos_enc(h)
        h = self.transformer(h, src_key_padding_mask=padding_mask)
        scores = (h @ self.attn_q) / (self.attn_q.shape[0] ** 0.5)
        if padding_mask is not None:
            scores = scores.masked_fill(padding_mask, float('-inf'))
        attn = torch.softmax(scores, dim=1)
        pooled = (attn.unsqueeze(-1) * h).sum(dim=1)
        logits = self.fc(self.drop(self.norm_out(pooled)))
        return (logits, attn) if return_weights else logits


HPARAMS = {'proj_dim':256,'num_layers':4,'num_heads':8,'ffn_dim':512,
           'dropout':0.1,'lr':3e-4,'weight_decay':1e-4,'batch_size':64}
MAX_EPOCHS, PATIENCE, WARMUP_EPOCHS = 30, 10, 3

print('Reading sequence lengths for length-sorted batching ...', flush=True)
ALL_LENS = np.zeros(len(COMB), dtype=np.int64)
_handles = {s: h5py.File(PATHS[s], 'r') for s in set(COMB_SRC)}
for i, (s, k) in enumerate(zip(COMB_SRC, COMB_KEY)):
    ALL_LENS[i] = min(_handles[s][str(k)].shape[0], MAX_SEQ_LEN)
for h in _handles.values(): h.close()
print('Lengths ready.')


def make_loader(positions, batch_size, shuffle):
    srcs, keys, labs = COMB_SRC[positions], COMB_KEY[positions], COMB_Y[positions]
    lengths = ALL_LENS[positions]
    ds = MultiH5Dataset(srcs, keys, labs, PATHS, MAX_SEQ_LEN)
    class _LenSampler(torch.utils.data.Sampler):
        def __init__(self): self.rng = np.random.default_rng(42)
        def __len__(self): return len(positions)
        def __iter__(self):
            order = np.argsort(lengths)
            batches = [list(order[i:i+batch_size]) for i in range(0, len(order), batch_size)]
            if shuffle: self.rng.shuffle(batches)
            for b in batches: yield from b
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, sampler=_LenSampler(),
        collate_fn=collate_variable_length, num_workers=0, pin_memory=True)

print('make_loader ready.')


In [ ]:
final_model = ESM3FerroCLF(**{k:HPARAMS[k] for k in
    ['proj_dim','num_layers','num_heads','ffn_dim','dropout']}).to(device)
n_params = sum(p.numel() for p in final_model.parameters())
print(f'ESM3FerroCLF  {n_params:,} parameters', flush=True)

opt   = torch.optim.AdamW(final_model.parameters(), lr=HPARAMS['lr'],
                          weight_decay=HPARAMS['weight_decay'])
crit  = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
scheduler = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[
    torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS),
    torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS-WARMUP_EPOCHS, eta_min=1e-6),
], milestones=[WARMUP_EPOCHS])

print('Building loaders ...', flush=True)
tr_dl  = make_loader(idx_train, HPARAMS['batch_size'], True)
val_dl = make_loader(idx_val,   64, False)
te_dl  = make_loader(idx_test,  64, False)
print('Loaders ready.\n', flush=True)

history, best_val_auc, best_state, wait = [], 0.0, None, 0
t_total = time.time()
print(f'{"Epoch":>5}  {"Loss":>9}  {"Val-AUC":>8}  {"LR":>9}  {"Time":>6}', flush=True)
for epoch in range(MAX_EPOCHS):
    t0 = time.time(); final_model.train(); run_loss = n_steps = 0
    for xb, yb, mask in tr_dl:
        xb, yb, mask = xb.to(device), yb.to(device), mask.to(device)
        opt.zero_grad()
        with torch.amp.autocast(device.type, enabled=use_amp):
            loss = crit(final_model(xb, mask), yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run_loss += loss.item(); n_steps += 1
    scheduler.step()
    final_model.eval(); vp, vt = [], []
    with torch.no_grad():
        for xb, yb, mask in val_dl:
            with torch.amp.autocast(device.type, enabled=use_amp):
                lo = final_model(xb.to(device), mask.to(device))
            vp.append(torch.softmax(lo,1)[:,1].cpu()); vt.append(yb)
    val_auc = roc_auc_score(torch.cat(vt).numpy(), torch.cat(vp).numpy())
    history.append({'epoch':epoch+1,'loss':run_loss/n_steps,'val_auc':val_auc,
                    'lr':scheduler.get_last_lr()[0]})
    flag = ''
    if val_auc > best_val_auc:
        best_val_auc = val_auc; wait = 0; flag = '  *'
        best_state = {k:v.clone() for k,v in final_model.state_dict().items()}
    else:
        wait += 1
        if wait >= PATIENCE: print(f'  Early stop at epoch {epoch+1}', flush=True); break
    print(f'  {epoch+1:3d}    {run_loss/n_steps:.4f}   {val_auc:.4f}'
          f'   {scheduler.get_last_lr()[0]:.2e}   {time.time()-t0:.0f}s{flag}', flush=True)
final_model.load_state_dict(best_state)
print(f'\nBest val AUC: {best_val_auc:.4f}  total {(time.time()-t_total)/60:.1f} min', flush=True)

# Gene-disjoint test (includes unseen hard-neg genes)
final_model.eval(); tp_, pr_, tt_ = [], [], []
with torch.no_grad():
    for xb, yb, mask in te_dl:
        with torch.amp.autocast(device.type, enabled=use_amp):
            lo = final_model(xb.to(device), mask.to(device))
        tp_.append(torch.softmax(lo,1)[:,1].cpu()); pr_.append(lo.argmax(1).cpu()); tt_.append(yb)
prob, pred, true = (torch.cat(tp_).numpy(), torch.cat(pr_).numpy(), torch.cat(tt_).numpy())
test_auc = roc_auc_score(true, prob)
print('\nHard-Negative Gene-Disjoint Test')
print(f'  Accuracy      : {accuracy_score(true, pred):.4f}')
print(f'  ROC-AUC       : {test_auc:.4f}   (random-split ESM3: 0.9552)')
print(f'  Avg Precision : {average_precision_score(true, prob):.4f}')

CKPT_PATH.parent.mkdir(exist_ok=True)
torch.save({'state_dict':best_state,'hparams':HPARAMS,'test_auc':test_auc,
            'split':'gene_disjoint_hardneg',
            'train_hardneg_genes':sorted(train_genes),
            'holdout_hardneg_genes':sorted(holdout_genes),
            'per_gene_cap':PER_GENE_CAP}, CKPT_PATH)
pd.DataFrame(history).to_csv(EMBED_DIR / 'training_history.csv', index=False)

# per-gene accuracy on the held-out test genes.
# te_dl is length-sorted, so predictions come back in argsort(length) order, NOT
# idx_test order — realign gene labels to the emission order (assert guards it).
_order = np.argsort(ALL_LENS[idx_test])
tg = COMB_GENE[idx_test][_order]
assert (COMB_Y[idx_test][_order] == true).all(), 'per-gene order realignment failed'
dfg = pd.DataFrame({'gene':tg,'true':true,'pred':pred,'prob':prob})
pg = (dfg.groupby('gene').apply(lambda x: pd.Series({
        'n':len(x),'acc':(x['pred']==x['true']).mean(),
        'label':x['true'].iloc[0],'mean_prob':x['prob'].mean()}))
      .reset_index().sort_values('acc'))
pg['kind'] = np.where(pg['label']==1, 'ferro+',
              np.where(pg['gene'].isin(set(rcd_train['gene'])), 'hard-neg', 'housekeeping-neg'))
pg.to_csv(EMBED_DIR / 'test_per_gene.csv', index=False)
print(f'Checkpoint → {CKPT_PATH}')
print('\nUNSEEN hard-negative (death) genes in the test split — specificity:')
_hn = pg[pg['kind']=='hard-neg']
print(_hn[['gene','n','acc','mean_prob']].to_string(index=False) if len(_hn)
      else '  (none landed in this test fold)')
print('\nHousekeeping negatives (context): mean acc '
      f"{pg.loc[pg['kind']=='housekeeping-neg','acc'].mean():.3f}")


## E · External validation — v4 (positive + hard-neg only)

The easy-negative / housekeeping tier is dropped. Two groups, neither seen in training:
- **ferroptosis genes** (`external/`) → **sensitivity**
- **30% held-out death genes** (hard negatives) → **specificity** (the 0.6294 number)


In [ ]:
# External validation — v4: positive + hard-neg ONLY (easy-neg tier dropped)
# The housekeeping / easy-negative probe is intentionally removed; we report the
# hard-negative death-gene specificity as THE external-validation specificity.
ext_records = []
for path in sorted(EXTERNAL_DIR.glob('*.xlsx')):          # ferroptosis positives
    if gene_from_fname(path.name) in EXCLUDE_POS: continue
    ext_records.extend(parse_xlsx(path, label=1, pathway='ferroptosis'))
extval = pd.DataFrame(ext_records)
extval = pd.concat([extval, rcd_hold], ignore_index=True)  # + held-out hard negatives (death genes)
extval = extval.groupby('gene', group_keys=False).apply(cap).reset_index(drop=True)
extval['h5key'] = ['E' + str(i) for i in range(len(extval))]
print(f'Validation set: {len(extval)} seqs  '
      f'({(extval["label"]==1).sum()} pos / {(extval["label"]==0).sum()} neg), '
      f'{extval["gene"].nunique()} genes  (positive + hard-neg; NO housekeeping)')

if EXTVAL_PATH.exists():
    print(f'Reusing cached {EXTVAL_PATH}')
else:
    embed_to_h5(extval['sequence'].tolist(), extval['h5key'].tolist(), EXTVAL_PATH)

# Inference with the freshly trained v4 model
clf = ESM3FerroCLF(**{k:HPARAMS[k] for k in
    ['proj_dim','num_layers','num_heads','ffn_dim','dropout']}).to(device)
clf.load_state_dict(torch.load(CKPT_PATH, map_location=device,
                               weights_only=False)['state_dict']); clf.eval()

ds = MultiH5Dataset(np.array(['ev']*len(extval)), extval['h5key'].to_numpy(),
                    extval['label'].to_numpy(), {'ev':str(EXTVAL_PATH)}, MAX_SEQ_LEN)
dl = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False,
        collate_fn=collate_variable_length, num_workers=0)
probs = []
with torch.no_grad():
    for xb, yb, mask in dl:
        with torch.amp.autocast(device.type, enabled=use_amp):
            lo = clf(xb.to(device), mask.to(device))
        probs.append(torch.softmax(lo,1)[:,1].cpu())
extval['prob'] = torch.cat(probs).numpy()
extval['pred'] = (extval['prob'] >= 0.5).astype(int)

# Metrics (positive -> sensitivity, hard-neg -> specificity)
y_true, y_pred = extval['label'].to_numpy(), extval['pred'].to_numpy()
sens = extval.loc[extval.label==1,'pred'].mean()
spec = 1 - extval.loc[extval.label==0,'pred'].mean()
print('\nExternal Validation — v4 (positive + hard-neg; easy-neg tier dropped)')
print(f'  Sequences   : {len(extval)}  ({(y_true==1).sum()} pos / {(y_true==0).sum()} neg)')
print(f'  Accuracy    : {accuracy_score(y_true, y_pred):.4f}')
print(f'  ROC-AUC     : {roc_auc_score(y_true, extval["prob"]):.4f}')
print(f'  Sensitivity : {sens:.4f}')
print(f'  Specificity : {spec:.4f}   <-- reported external-validation specificity')

per_gene = (extval.groupby('gene').apply(lambda x: pd.Series({
        'label':x['label'].iloc[0],'n':len(x),
        'rate':(x['pred']==x['label']).mean(),'mean_prob':x['prob'].mean()}))
        .reset_index())
per_gene['tier']   = np.where(per_gene['label']==1, 'positive (ferroptosis)', 'hard-neg (death)')
per_gene['metric'] = np.where(per_gene['label']==1, 'sensitivity', 'specificity')
per_gene.to_csv(V4_DIR / 'v4_extval_per_gene.csv', index=False)
extval[['gene','label','prob','pred']].to_csv(V4_DIR / 'v4_extval_predictions.csv', index=False)
print('\nPositive genes (sensitivity):')
print(per_gene[per_gene.label==1].sort_values('rate', ascending=False)
      [['gene','n','rate','mean_prob']].to_string(index=False))
print('\nHard-negative death genes (specificity):')
print(per_gene[per_gene.label==0].sort_values('rate', ascending=False)
      [['gene','n','rate','mean_prob']].to_string(index=False))
print(f'\nSaved -> {V4_DIR / "v4_extval_per_gene.csv"} and v4_extval_predictions.csv')
